### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_UJZSQRXZAIGC


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250212_squid_cal520"
recname = "rec3"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250212_rec3"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_transformed_corrected_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_transformed_corrected_renamed/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(865000,) (13164,) (72,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(114, 865000)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.173877,-0.276452,0.006041,-0.031326,0.216353,0.301250,-0.217460,-0.264488,...,-0.097114,-0.450120,-0.404607,-0.436258,-0.382514,-0.539513,-0.583700,-0.394905,-0.242419,-0.255327
1,0,1,0.160342,-0.421678,-0.751778,-0.748718,-0.719448,-0.828465,-0.650186,-0.433620,...,0.098129,-0.216258,0.259363,0.159666,-0.211428,-0.073327,-0.286631,-0.368334,-0.610172,-0.195606
2,0,2,0.380029,-0.458869,-0.870456,-0.830386,-0.837839,-1.101920,-0.854493,-0.796436,...,0.320422,0.947310,0.890295,0.534838,0.318240,0.512992,0.507710,0.447372,0.312557,0.668886
3,0,3,0.452343,-0.204476,-0.351233,-0.316856,-0.157273,-0.306451,-0.549750,-0.112784,...,0.350886,0.447505,0.809955,0.653986,0.356431,0.676946,0.906292,0.973534,0.864004,0.609254
4,0,4,0.119419,-0.548152,-0.538794,-0.800542,-1.149341,-0.840095,-0.459621,-0.296091,...,-0.028067,-0.161605,-0.077929,-0.113106,0.052951,-0.189643,0.259677,0.392791,0.243255,0.408605
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,0,109,0.448818,-1.028552,-0.577987,-0.810328,-0.784290,-0.448615,-0.984239,-0.962707,...,0.752040,0.367040,0.008096,0.616152,0.747495,0.456608,0.743449,0.762001,0.630281,0.506442
110,0,110,0.222841,0.331949,0.077692,0.506042,0.759369,0.732622,0.522251,0.461508,...,-0.030622,-0.138031,0.321367,0.264331,0.102445,0.341249,0.334461,0.607068,0.305565,0.246871
111,0,111,0.126788,-0.079862,0.281924,0.273820,0.099264,0.059534,0.124512,0.437739,...,-0.573211,-0.217843,0.302163,0.430731,-0.244160,-0.529516,-0.240373,-0.077659,0.259490,-0.081126
112,0,112,0.441971,-0.972952,-0.577740,-0.735946,-0.660088,-0.444821,-0.792654,-0.654284,...,0.566916,1.014992,0.423155,0.441095,0.864240,0.486505,1.046964,0.851038,0.415452,0.731989


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250212_rec3"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_UJZSQRXZAIGC/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250212_rec3"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:1
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_UJZSQRXZAIGC/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250212_rec3"', returncode=0)